# Windows Security Log Demo

Simple, non-streaming demo: read the JSON file produced by `winlog_demo.ps1`
and show it as a table. No Auto Loader, no checkpoints, no streaming — just
a batch read, a couple of summaries, and an optional table save.

Edit the two widgets below to point at a catalog/schema/volume you already
have access to — no new catalog or schema needs to be created for this.
The file itself can arrive either via `databricks fs cp` or by dragging it
into the volume through the Databricks UI.

In [ ]:
dbutils.widgets.text("volume_path", "/Volumes/staging_monthly/winlog/landing", "Volume path (edit me)")
dbutils.widgets.text("table_name", "staging_monthly.winlog.security_demo", "Destination table (edit me)")

In [ ]:
volume_path = dbutils.widgets.get("volume_path")
table_name  = dbutils.widgets.get("table_name")
file_path   = f"{volume_path}/winlog_demo_security_log.json"

### Read the file
One JSON object per line (newline-delimited JSON) — Spark reads this shape natively.

In [ ]:
df = spark.read.json(file_path)
display(df)

### A couple of quick summaries
The kind of rollup a Splunk/qmulos dashboard would show.

In [ ]:
display(df.groupBy("level").count().orderBy("count", ascending=False))

In [ ]:
display(df.groupBy("event_id", "task_display_name").count().orderBy("count", ascending=False))

### Save as a table (optional)
Plain batch overwrite — no merge/upsert logic needed for a one-off demo.

In [ ]:
df.write.mode("overwrite").saveAsTable(table_name)
display(spark.sql(f"SELECT * FROM {table_name} LIMIT 20"))